# 01 — Exploratory Data Analysis (EDA)

Khám phá tổng quan dataset Superstore Sales:
- Thống kê mô tả
- Phân phối doanh số
- Phân tích theo Category, Segment, Region
- Xu hướng doanh số theo thời gian


In [ ]:
# %% [import] Thư viện và cấu hình
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_params, load_raw_data, get_data_info

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
# %% [load] Đọc dữ liệu thô từ data/raw/train.csv
params = load_params()
df = load_raw_data(params)
df.head()


## 1. Thống kê tổng quan

In [ ]:
# %% [info] Thống kê tổng quan: shape, date range, missing, duplicates
info = get_data_info(df)
print(f"Shape       : {info['shape']}")
print(f"Date range  : {info['date_range'][0].date()} → {info['date_range'][1].date()}")
print(f"Orders      : {info['n_orders']}")
print(f"Customers   : {info['n_customers']}")
print(f"Products    : {info['n_products']}")
print(f"Duplicates  : {info['duplicates']}")
print(f"\nMissing values:")
print({k: v for k, v in info['missing'].items() if v > 0} or "Không có")

In [ ]:
# %% [describe] Thống kê mô tả các cột số
df.describe()

## 2. Phân phối doanh số (Sales)

In [ ]:
# %% [plot] Histogram + Boxplot phân phối Sales, phát hiện outlier
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["Sales"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Phân phối Sales")
axes[0].set_xlabel("Sales (USD)")

sns.boxplot(x=df["Sales"], ax=axes[1], color="coral")
axes[1].set_title("Boxplot Sales")

plt.tight_layout()
plt.savefig("../outputs/figures/sales_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Doanh số theo Category và Sub-Category

In [ ]:
# %% [plot] Barplot doanh số theo Category và Sub-Category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
sns.barplot(x=cat_sales.values, y=cat_sales.index, ax=axes[0], palette="Blues_r")
axes[0].set_title("Tổng doanh số theo Category")
axes[0].set_xlabel("Sales (USD)")

subcat_sales = df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False)
sns.barplot(x=subcat_sales.values, y=subcat_sales.index, ax=axes[1], palette="Greens_r")
axes[1].set_title("Tổng doanh số theo Sub-Category")
axes[1].set_xlabel("Sales (USD)")

plt.tight_layout()
plt.savefig("../outputs/figures/sales_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Doanh số theo Segment và Region

In [ ]:
# %% [plot] Barplot doanh số theo Segment và Region
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

seg_sales = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False)
sns.barplot(x=seg_sales.index, y=seg_sales.values, ax=axes[0], palette="Oranges_r")
axes[0].set_title("Tổng doanh số theo Segment")
axes[0].set_ylabel("Sales (USD)")

reg_sales = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
sns.barplot(x=reg_sales.index, y=reg_sales.values, ax=axes[1], palette="Purples_r")
axes[1].set_title("Tổng doanh số theo Region")
axes[1].set_ylabel("Sales (USD)")

plt.tight_layout()
plt.savefig("../outputs/figures/sales_by_segment_region.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Xu hướng doanh số theo thời gian

In [ ]:
# %% [plot] Line chart doanh số theo tháng 2015–2018
from src.features import build_monthly_sales

monthly = build_monthly_sales(df)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly["YearMonth"], monthly["Sales"], marker="o", color="steelblue", linewidth=2)
ax.fill_between(monthly["YearMonth"], monthly["Sales"], alpha=0.15, color="steelblue")
ax.set_title("Doanh số theo tháng (2015–2018)")
ax.set_xlabel("Tháng")
ax.set_ylabel("Sales (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../outputs/figures/monthly_sales.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Phân tích theo Ship Mode

In [ ]:
# %% [plot] Số đơn và doanh số TB theo Ship Mode
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ship_count = df["Ship Mode"].value_counts()
sns.barplot(x=ship_count.index, y=ship_count.values, ax=axes[0], palette="Blues_r")
axes[0].set_title("Số đơn hàng theo Ship Mode")
axes[0].set_ylabel("Số đơn hàng")

ship_sales = df.groupby("Ship Mode")["Sales"].mean().sort_values(ascending=False)
sns.barplot(x=ship_sales.index, y=ship_sales.values, ax=axes[1], palette="Oranges_r")
axes[1].set_title("Doanh số TB theo Ship Mode")
axes[1].set_ylabel("Sales trung bình (USD)")

plt.tight_layout()
plt.savefig("../outputs/figures/shipmode_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Tương quan giữa các biến số

In [ ]:
# %% [plot] Heatmap tương quan Sales, Shipping Days, Month, Year
df_num = df[["Sales"]].copy()
df_num["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df_num["Month"] = df["Order Date"].dt.month
df_num["Year"]  = df["Order Date"].dt.year

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df_num.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.savefig("../outputs/figures/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()